# Backtesting and Training-Window Sensitivity

A single holdout can give a misleading view of forecasting quality. This notebook documents the second stage of the case study: repeated 7-day backtests and sensitivity to the amount of training history retained by the local Prophet model.

## Backtest design

Each evaluation predicts the next **168 hourly observations**. Forecast windows are non-overlapping and all metrics are calculated out of sample.

The original local-model experiment used:

- 28 days / 672 hours of rolling training history;
- 7 days / 168 hours of forecast horizon;
- a 7-day step between forecast origins;
- MAE, MSE, wMAPE, and signed bias.

Across 13 forecast windows for a representative high-volume FOOD series, wMAPE varied materially (roughly 32%–46%) and bias changed sign. This demonstrates why a single train/test split is insufficient for judging operational stability.

In [ ]:
import numpy as np
import pandas as pd

def wmape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.abs(y_true).sum()
    return np.nan if denom == 0 else np.abs(y_true - y_pred).sum() / denom

def bias(y_true, y_pred):
    return np.mean(np.asarray(y_pred) - np.asarray(y_true))

## Training-window sensitivity

The next question was whether the original 28-day rolling window discarded useful recurring demand information. The model, target, horizon, preprocessing, and metrics were held fixed while the training history was increased to 4, 6, and 8 weeks.

Because longer histories cannot generate the earliest folds, comparisons were restricted to **common forecast windows**. This prevents confounding training-window effects with differences in evaluation dates.

In [ ]:
window_summary = pd.DataFrame({
    "Training history": ["4 weeks", "6 weeks", "8 weeks"],
    "Mean wMAPE": [0.3889, 0.3783, 0.3707],
    "Mean absolute bias": [1.98, 1.75, 1.48],
})

window_summary.style.format({
    "Mean wMAPE": "{:.2%}",
    "Mean absolute bias": "{:.2f}",
})

## Interpretation

Longer training history improved both forecast accuracy and bias stability on the common evaluation windows:

| Training history | Mean wMAPE | Mean absolute bias |
|---|---:|---:|
| 4 weeks | 38.89% | 1.98 |
| 6 weeks | 37.83% | 1.75 |
| 8 weeks | **37.07%** | **1.48** |

The improvement is incremental rather than dramatic, but consistent. The experiment suggests that the 28-day window was unnecessarily aggressive for a series with recurring weekly/intraday structure.

More broadly, the result illustrates an important forecasting practice: **training-window length is itself a modeling assumption and should be validated**, especially when non-stationarity is being balanced against the value of repeated seasonal history.

## Relationship to the main notebook

This analysis serves a different purpose from the champion/challenger comparison:

- **This notebook:** tests temporal stability and training-history sensitivity for the local forecasting workflow.
- **Main notebook:** compares Seasonal Naive, local Prophet, and Global XGBoost on identical rolling-origin windows across the full Hub × Segment portfolio.

Together they separate two questions: *is a model stable over time?* and *which model is strongest on a fair common evaluation?*